### Imports

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from models.archs.utils import init_model
from unlearn.GA import GA
from data.utils import split_forget_retain
# from trainer.utils import get_memory_footprint, get_model_weight_norm
# from evaluation.utils import check_accuracy

### Set configs for the experiment

In [2]:
device = "mps" if torch.mps.is_available() else "cpu"
exp_config = {

    "description": "FULL EXPERIMENT WITH UPDATES - forget acc vs retain acc, using GA",
    
    "device": device,
    "model_class": "CIFARNET",
    "num_runs": 3,
    "retrain_from_scratch": True,
    "train_base": True,

    "data": {
        "batch_size": 256,
        "num_workers": 0,
        },

    "training": {
        "num_epochs": 30,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "lr_scheduler_params": {
            "factor": 0.5,
            "patience": 2,
            "threshold": .01,
            },
        "batch_print_freq": 150,
        },
    
    "unlearning": {
        "methods": ["GA"],
        "metrics": ["forget_acc", "retain_acc"],
        "num_epochs": 12,
        "measure_every": 1,
        "save_checkpoints_at": [1, 6, 12],
        "classes_to_unlearn": [0],
        "percents_to_unlearn": None,
        "learning_rate": {
            "GA": 5e-5, # recall we are now doing SGD, so the learning rate is different from Adam
            }
        },
}

### The GA  regimen

In [3]:
import json
import numpy as np

def GA_regimen(model, dataloaders, metrics_to_use, num_epochs, criterion, opt, print_freq, measure_every, device, name, results_folder, wandb_log, save_checkpoints_at, checkpoint_subfolder):

    # create save folder if it doesn't exist
    if not os.path.exists(results_folder):
        print(f"{results_folder} doesn't exist - creating it...\n")
        os.makedirs(results_folder, exist_ok=True)

    # For each epoch ...
    for k in range(1, num_epochs+1):

        # ... do one round of GA,
        model, top1_avg = GA(dataloaders["forget_train"], model, criterion, opt, epoch = k, print_freq = print_freq, device = device)

        # ... run some evaluations
        results = measure_unlearning_metrics(name = name, epoch = k, metrics = metrics_to_use, model = model, dataloaders = dataloaders, device = device)
        results["type"] = "unlearn"

        # ... print a subset of these results
        print(f"Epoch {k} Results:")
        filtered_results = {k: f"{results[k]:.4f}" for k in metrics_to_use if k in results}
        print(json.dumps(filtered_results, indent=4), "\n")

        # ... and if we're saving results out,
        if k % measure_every == 0:
            
            # ... create an epoch-level subfolder if it doesn't exist
            epoch_results_folder = os.path.join(results_folder, f"epoch_{k}")
            if not os.path.exists(epoch_results_folder):   
                print(f"{epoch_results_folder} doesn't exist - creating it...\n")
                os.makedirs(epoch_results_folder, exist_ok=True)

            # ... and save them
            if wandb_log:
                wandb.log(results)
            with open(os.path.join(epoch_results_folder, f"{name}.json"), "w") as f:
                json.dump(results, f, indent=4)

        # ... and if we're saving checkpoints out,
        if k in save_checkpoints_at:
            unlearn_checkpoint_path = os.path.join(checkpoint_subfolder, f"GA_epoch_{k}_{name}.pth")
            checkpoint = {
                'epoch': k,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': opt.state_dict()
            }
            torch.save(checkpoint, unlearn_checkpoint_path, _use_new_zipfile_serialization = False)


    return model

In [4]:
def do_unlearning(
        base_results_folder,
        config,
        method,
        base_model,
        dataloaders,
        name,
        wandb_log,
        save_checkpoints_at,
        checkpoint_subfolder
        ):
    
    # ... and do the method in question
    if method == "GA":
        
        _ = GA_regimen(
            base_model, 
            dataloaders = dataloaders, 
            metrics_to_use=config["unlearning"]["metrics"],
            num_epochs = config["unlearning"]["num_epochs"], 
            criterion = nn.CrossEntropyLoss(), 
            opt = optim.SGD(base_model.parameters(), lr=config["unlearning"]["learning_rate"]["GA"]),
            print_freq = 15, 
            measure_every = config["unlearning"]["measure_every"], 
            device = config["device"], 
            name = name, 
            results_folder = f"{base_results_folder}/GA",
            wandb_log = wandb_log,
            save_checkpoints_at = save_checkpoints_at,
            checkpoint_subfolder = checkpoint_subfolder
            )
        


### Protocol for several runs

In [5]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/jerrymoncus/.netrc.
wandb: Currently logged in as: jjmoncus (jjmoncus706) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
import random
from models.archs.utils import init_model
from torch.optim.lr_scheduler import ReduceLROnPlateau
from trainer.utils import training_regimen
from data.dataloaders import cifar10_dataloaders
from evaluation.utils import measure_unlearning_metrics

def run_experiment(config, results_folder, checkpoint_folder):

    # init wandb
    wandb.init(
      project="Verifying-Unlearning-2026",
      name=f"Grand_Seed_{config['GRAND_SEED']}",
      config=config,
      reinit= "finish_previous"
    )

    
    print("="*70)
    print("="*19 + "  " + f"RUNNING EXPERIMENT, SEED {config["GRAND_SEED"]}" + "  " + "="*19)
    print("="*70 + "\n")

    # Make experiment results folder if it doesnt already exist
    if not os.path.exists(results_folder):
        print(f"{results_folder} doesn't exist - creating it...\n")
        os.makedirs(results_folder, exist_ok=True)

    # Save the config for this experiment to the main results folder
    with open(os.path.join(results_folder, "experiment_config.json"), "w") as f:
        json.dump(config, f, indent=4)

    # create a subfolder for saving model checkpoints for this experiment
    print(f"All models will be of class {config["model_class"]}.\n")
    checkpoint_subfolder = os.path.join(checkpoint_folder, f"seed_{config["GRAND_SEED"]}")
    if not os.path.exists(checkpoint_subfolder):   
        print(f"{checkpoint_subfolder} doesn't exist - creating it...\n")
        os.makedirs(checkpoint_subfolder, exist_ok=True)

    # ----------------------------------------------------------------------------------- #
    # ----------------------------------------------------------------------------------- #
    # ---------------- TRAIN A BASE MODEL, FROM WHICH UNLEARNING BEGINS ----------------- #
    # ----------------------------------------------------------------------------------- #
    # ----------------------------------------------------------------------------------- #

    # If you want to train your base model, ...
    if config["train_base"]: 

        print("-"*57)
        print("-"*13 + "  " + f"TRAINING NEW BASE MODEL" + "  " + "-"*13)
        print("-"*57 + "\n")

        # get some data
        full_train, full_val, full_test = cifar10_dataloaders(
            data_dir="data/CIFAR10", 
            batch_size=config["data"]["batch_size"], 
            num_workers=config["data"]["num_workers"], 
            seed=config["GRAND_SEED"], 
            class_to_replace=None, 
            percent_to_replace=None
            )
        

        # init model, opt, criterion, and scheduler
        empty_model = init_model(model_class = config["model_class"]).to(config["device"])
        opt = optim.Adam(empty_model.parameters(), lr=config["training"]["learning_rate"], weight_decay = config["training"]["weight_decay"])
        criterion = nn.CrossEntropyLoss()
        scheduler = ReduceLROnPlateau(opt, mode='min', 
                                    factor=config["training"]["lr_scheduler_params"]["factor"], 
                                    patience=config["training"]["lr_scheduler_params"]["patience"], 
                                    threshold = config["training"]["lr_scheduler_params"]["threshold"], 
                                    threshold_mode = "rel")
        
        # train
        base_model_path = os.path.join(checkpoint_subfolder, "base_model.pth")
        _, opt, scheduler = training_regimen(
            empty_model, 
            full_train, 
            full_val, 
            opt, 
            criterion, 
            scheduler, 
            device = config["device"], 
            num_epochs=config["training"]["num_epochs"], 
            model_path = base_model_path,
            print_freq = config["training"]["batch_print_freq"])

    # Otherwise, pull a good base model from somewhere
    else:
        
        print("-"*57)
        print("-"*5 + "  " + f"NOT TRAINING BASE MODEL - PULLING INSTEAD" + "  " + "-"*5)
        print("-"*57 + "\n")

        base_model_path = "models/model_checkpoints/VGG_base_model.pth"

    # Regardless, init a fresh copy of the base model (we will use this one just to evaluate our metrics within each run and class)
    base_model = init_model(model_class = config["model_class"], checkpoint_path = base_model_path).to(config["device"])
    
    # and init a subfolder for all results pertaining to the base model
    base_subfolder = os.path.join(results_folder, "base")
    if not os.path.exists(base_subfolder):   
        print(f"{base_subfolder} doesn't exist - creating it...\n")
        os.makedirs(base_subfolder, exist_ok=True)


    print("-"*54)
    print("-"*15 + "  " + f"BEGINNING UNLEARNING" + "  " + "-"*15)
    print("-"*54 + "\n")
    # For some number of runs, ...
    for i in range(1, config["num_runs"]+1):
            
        print("="*25 + "    " + f"RUN {i}\n")

        # ... decide if we're unlearning percents or classes (whichever one is non-empty)
        we_are_unlearning_classes = True if config["unlearning"]["classes_to_unlearn"] else False
        items_to_unlearn = config["unlearning"]["classes_to_unlearn"] if we_are_unlearning_classes else config["unlearning"]["percents_to_unlearn"]
        if not items_to_unlearn:
            raise ValueError("Either `classes_to_unlearn` or `percents_to_unlearn` need to be specified")
        
        # ... Loop through all the items we want to unlearn, 
        for c in items_to_unlearn:

            # ... set an item seed (either with class or percent appended)
            banner = f"RUN {i}, CLASS {c}" if we_are_unlearning_classes else f"RUN {i}, PERCENT {c}"
            print("-"*15 + "    " + banner + "\n")
            micro_name = f"run_{i}_class_{c}" if we_are_unlearning_classes else f"run_{i}_percent_{c}"
            micro_seed = int(f"{config["GRAND_SEED"]}{i}{c}")

            # ----------------------------------------------------------------------------------- #
            # ----------------------------------------------------------------------------------- #
            # ------------------------------- DO SOME UNLEARNING -------------------------------- #
            # ----------------------------------------------------------------------------------- #
            # ----------------------------------------------------------------------------------- #


            # ... be intelligent about setting `class_to_replace` or `percent_to_replace` if either is None
            class_param = c if we_are_unlearning_classes else None
            print(f"Class param = {class_param}")
            percent_param = c if not we_are_unlearning_classes else None
            print(f"Percent param = {percent_param}")

            # ... get some unlearning data for this particular run
            marked_train, marked_val, marked_test = cifar10_dataloaders(
                data_dir="data/CIFAR10", 
                batch_size=config["data"]["batch_size"], 
                num_workers=config["data"]["num_workers"], 
                seed = micro_seed, 
                class_to_replace=class_param, 
                percent_to_replace=percent_param, 
                only_mark=True
                )
            forget_train, retain_train = split_forget_retain(marked_train, batch_size=config["data"]["batch_size"], seed=micro_seed)
            forget_val, retain_val = split_forget_retain(marked_val, batch_size=config["data"]["batch_size"], seed=micro_seed)
            forget_test, retain_test = split_forget_retain(marked_test, batch_size=config["data"]["batch_size"], seed=micro_seed)
            unlearning_loaders = {
                "forget_train": forget_train, 
                "retain_train": retain_train, 
                "forget_val": forget_val, 
                "retain_val": retain_val,
                "forget_test": forget_test, 
                "retain_test": retain_test
            }

            # ... evaluate how good your base model is, w.r.t. the data loaders for this particular run and class/percent, and save results
            # ------- the base model needs to be evaluated for every draw of "forget set", since it might be a new random subsample every time
            # ------- in the case of class forgetting, we could I guess just measure this once, since accuracy isn't dependent on the order of the dataloader.
            base_name = f"base_{micro_name}"
            base_results = measure_unlearning_metrics(
                name = base_name, 
                epoch = None, 
                metrics = config["unlearning"]["metrics"], 
                model = base_model, 
                dataloaders = unlearning_loaders, 
                device = config["device"]
                )
            base_results["type"] = "base"
            
            # ... save base results
            wandb.log(base_results)
            with open(os.path.join(base_subfolder, f"{base_name}.json"), "w") as f:
                json.dump(base_results, f, indent=4)

            # ... and then apply each unlearning method over this particular data, and save their results in their subfolder.
            for method in config["unlearning"]["methods"]:
                
                # ... we need a new copy of the base model to begin unlearning each method on.
                unlearn_model = init_model(model_class = config["model_class"], checkpoint_path = base_model_path).to(config["device"])
                # ... has to be in eval mode I think (so BarchNorm layers aren't screwed)
                unlearn_model.eval()
                # ... actually doing the unlearning (results are saved out underneath this function)
                item_name = f"class_{c}" if we_are_unlearning_classes else f"percent_{c}"
                _ = do_unlearning(
                    base_results_folder = f"{results_folder}/unlearn/run_{i}",
                    config = config,
                    method = method,
                    base_model = unlearn_model,
                    dataloaders = unlearning_loaders,
                    name = item_name,
                    wandb_log = True,
                    save_checkpoints_at = config["unlearning"]["save_checkpoints_at"],
                    checkpoint_subfolder = checkpoint_subfolder
                    )

            # ----------------------------------------------------------------------------------- #
            # ----------------------------------------------------------------------------------- #
            # ----------------------------- RETRAIN FROM SCRATCH -------------------------------- #
            # ----------------------------------------------------------------------------------- #
            # ----------------------------------------------------------------------------------- #

            if config["retrain_from_scratch"]:
                appendage = f"class {c}" if we_are_unlearning_classes else f"percent {c}"
                print(f" ----- Retraining from scratch for run {i}, {appendage} ----- \n")
                # init model, opt, criterion, and scheduler
                empty_model = init_model(model_class = config["model_class"]).to(config["device"])
                opt = optim.Adam(empty_model.parameters(), lr=config["training"]["learning_rate"], weight_decay = config["training"]["weight_decay"])
                criterion = nn.CrossEntropyLoss()
                scheduler = ReduceLROnPlateau(opt, mode='min', 
                                            factor=config["training"]["lr_scheduler_params"]["factor"], 
                                            patience=config["training"]["lr_scheduler_params"]["patience"], 
                                            threshold = config["training"]["lr_scheduler_params"]["threshold"], 
                                            threshold_mode = "rel")
                # train
                retrain_name = f"retrain_{micro_name}"
                retrain_checkpoint_path = os.path.join(checkpoint_subfolder, f"{retrain_name}.pth")
                _, opt, scheduler = training_regimen(
                    empty_model, 
                    retain_train, 
                    retain_val, 
                    opt, 
                    criterion, 
                    scheduler, 
                    device = config["device"], 
                    num_epochs=config["training"]["num_epochs"], 
                    model_path = retrain_checkpoint_path,
                    print_freq = config["training"]["batch_print_freq"])
                
                # pull the best model from the checkpoint, and eval on metrics
                retrained_model = init_model(model_class = config["model_class"], checkpoint_path = retrain_checkpoint_path).to(config["device"])
                retrained_results = measure_unlearning_metrics(name = retrain_name, epoch = None, metrics = config["unlearning"]["metrics"], model = retrained_model, dataloaders = unlearning_loaders, device = config["device"])
                retrained_results["type"] = "retrain"

                # and init a subfolder for all results pertaining to the retrained models
                retrain_subfolder = os.path.join(results_folder, "retrain")
                if not os.path.exists(retrain_subfolder):   
                    print(f"{retrain_subfolder} doesn't exist - creating it...\n")
                    os.makedirs(retrain_subfolder, exist_ok=True)

                # save retrain results
                wandb.log(retrained_results)
                with open(os.path.join(retrain_subfolder, f"{retrain_name}.json"), "w") as f:
                    json.dump(retrained_results, f, indent=4)

            
    wandb.finish()

    print("-"*70)
    print("-"*19 + "  " + f"FINISHED EXPERIMENT, SEED {config["GRAND_SEED"]}" + "  " + "-"*19)
    print("-"*70 + "\n")
    

### Check metrics on unlearned models

In [7]:
# MAKE A RANDOM SEED
exp_config["GRAND_SEED"] = 2
# DO EXP
run_experiment(config = exp_config, results_folder = f"results/seed_{exp_config["GRAND_SEED"]}", checkpoint_folder="models/model_checkpoints")

===================  RUNNING EXPERIMENT, SEED 2  ===================

All models will be of class CIFARNET.

---------------------------------------------------------
-------------  TRAINING NEW BASE MODEL  -------------
---------------------------------------------------------



/opt/miniconda3/envs/comp0081/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


========== DATALOADER INFO
Dataset: CIFAR-10
Train: 45000 images for training
Val: 5000 images for validation
Test: 10000 images for testing
Training augmentation = randomcrop(32,4) + randomhorizontalflip + colorjitter + randomrotation + normalize
Validation/Test augmentation = normalize


 ----- EPOCH 1 ----- 

Batch 0: Loss = 2.3015
Batch 150: Loss = 1.6829
Epoch 1 | LR: 1.0e-03 | Val Loss: 1.4743, Val Acc: 0.45 | RAM: 0.50GB | VRAM: 1.14GB | Weight Norm: 18.618
--- Epoch 1: New best model saved! ---

 ----- EPOCH 2 ----- 

Batch 0: Loss = 1.3989
Batch 150: Loss = 1.3095
Epoch 2 | LR: 1.0e-03 | Val Loss: 1.2942, Val Acc: 0.53 | RAM: 0.49GB | VRAM: 1.14GB | Weight Norm: 21.644
--- Epoch 2: New best model saved! ---

 ----- EPOCH 3 ----- 

Batch 0: Loss = 1.2982
Batch 150: Loss = 1.1340
Epoch 3 | LR: 1.0e-03 | Val Loss: 1.0995, Val Acc: 0.59 | RAM: 0.48GB | VRAM: 1.14GB | Weight Norm: 24.628
--- Epoch 3: New best model saved! ---

 ----- EPOCH 4 ----- 

Batch 0: Loss = 1.0096
Batch 150

/opt/miniconda3/envs/comp0081/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch: [1][0/18]	Loss -0.3213 (-0.3213)	Accuracy 90.625 (90.625)	Time 0.11
Epoch: [1][15/18]	Loss -0.3081 (-0.2857)	Accuracy 89.453 (90.308)	Time 1.40
Epoch 1 Results:
{
    "forget_acc": "0.9018",
    "retain_acc": "0.8342"
} 

Epoch: [2][0/18]	Loss -0.3832 (-0.3832)	Accuracy 85.938 (85.938)	Time 0.09
Epoch: [2][15/18]	Loss -0.4038 (-0.3137)	Accuracy 84.766 (89.551)	Time 1.37
Epoch 2 Results:
{
    "forget_acc": "0.8927",
    "retain_acc": "0.8354"
} 

results/seed_2/unlearn/run_1/GA/epoch_2 doesn't exist - creating it...

Epoch: [3][0/18]	Loss -0.2363 (-0.2363)	Accuracy 93.750 (93.750)	Time 0.09
Epoch: [3][15/18]	Loss -0.3362 (-0.3300)	Accuracy 89.062 (88.989)	Time 1.33
Epoch 3 Results:
{
    "forget_acc": "0.8809",
    "retain_acc": "0.8370"
} 

results/seed_2/unlearn/run_1/GA/epoch_3 doesn't exist - creating it...

Epoch: [4][0/18]	Loss -0.2659 (-0.2659)	Accuracy 91.016 (91.016)	Time 0.09
Epoch: [4][15/18]	Loss -0.4481 (-0.3580)	Accuracy 87.891 (87.939)	Time 1.42
Epoch 4 Results:
{

epoch,▁▂▂▃▄▄▅▅▆▇▇█▁▂▂▃▄▄▅▅▆▇▇█▁▂▂▃▄▄▅▅▆▇▇█
forget_acc,██████▇▇▇▇▆▅▄▁██████▇▇▇▆▆▄▁██████▇▇▇▇▆▆▁
retain_acc,▇▇▇▇▇▇▇▇█▇▇▅▁▇▇▇▇▇▇▇██▇▇▆▂█▇▇▇▇▇▇▇██▇▇▆▇
forget_acc,0
name,retrain_run_3_class_...
retain_acc,0.83722
type,retrain


----------------------------------------------------------------------
-------------------  FINISHED EXPERIMENT, SEED 2  -------------------
----------------------------------------------------------------------

